# Dialog Chrome, Navigation, Launcher & Accessibility (`CRAFT.js`, `CRAFT.css`)

> Page-level UI chrome around Solveit dialogs: mobile/desktop mode detection (`window.craftMobile`), CSS auto-hiding top navbar, compact Monaco line height, `Mod+Shift+P` command/file launcher, CSS-hidden bottom Monaco composer (`window.craftIdleComposer`, `Shift+I` toggle), and tooltip-to-`aria-label` accessibility pass (`window.craftA11y`).

`window.craftMobile` runs here early in `CRAFT.js` so `html.craft-mobile` is applied before cell presentation, insertion, and editor modules mount.

## Chrome Templates (`#| export_ft`)

Icons and static DOM shells for the mobile/desktop view toggle (`#craft-mobile-toggle`) and the command launcher modal (`#solveit-launcher`).

In [ ]:
#| export_ft CRAFT.js
#| eval: false
TEMPLATES = dict(
    mobile_toggle_btn=Button(
        svg_icon([ft('rect', x='7', y='2', width='10', height='20', rx='2'), 'M11 18h2'], size=15, sw=1.8, cls='craft-icon-phone'),
        svg_icon([ft('rect', x='2', y='3', width='20', height='14', rx='2'), 'M8 21h8M12 17v4'], size=15, sw=1.8, cls='craft-icon-desk'),
        id='craft-mobile-toggle', klass='craft-nav-icon-btn', type='button'),
    launcher_root=craft_overlay(
        craft_card(
            Input(type='text', placeholder='Run a command or open a dialog/file…',
                  autocomplete='off', spellcheck='false', aria_label='Search commands and files', data_ref='input'),
            Ul(role='listbox', data_ref='list'),
            cls='solveit-launcher-card'),
        id='solveit-launcher', aria_label='Command palette'),
)

## Mobile Mode Detection (`window.craftMobile`)

Auto-enables `html.craft-mobile` on touch-only devices (`(hover: none) and (pointer: coarse)`) while letting desktop users toggle mobile view explicitly via `#craft-mobile-toggle`, the launcher, or `window.craftMobile.setMode('auto' | 'mobile' | 'desktop')`.

In [ ]:
#| export_js CRAFT.js
#| eval: false
/* ── Mobile mode detection & view toggle ───────────────────────────────
   Auto-enables `html.craft-mobile` only on touch/mobile devices (`(hover: none)
   and (pointer: coarse)`), so shrinking a desktop window stays in desktop view.
   Users can explicitly switch between Desktop and Mobile at any time via the
   navbar toggle button (#craft-mobile-toggle), the launcher, or
   `window.craftMobile.setMode('auto'|'mobile'|'desktop')`. */
(() => {
  window.craftMobile?.disable?.();
  if (!window.__craftCMCore.features.init('mobile-mode', 'Mobile mode toggle', 'Auto-detect touch layout & view switcher')) return;
  const {mountTpl, qs, on, store, patch} = window.__craftCMCore;
  const ac = new AbortController(), sig = {signal: ac.signal};
  const PREF_KEY = 'craft-mobile-mode';
  const mq = matchMedia('(hover: none) and (pointer: coarse)');
  const root = document.documentElement, listeners = new Set();
  let meta = qs('meta[name="viewport"]'), addedMeta = false, prevMeta = meta?.content;
  const VIEWPORT = 'width=device-width, initial-scale=1, viewport-fit=cover, interactive-widget=resizes-content';
  if (!meta) { meta = document.createElement('meta'); meta.name = 'viewport'; document.head.append(meta); addedMeta = true; }
  if (!/width\s*=\s*device-width/.test(meta.content || '')) meta.content = VIEWPORT;
  else {
    if (!/viewport-fit/.test(meta.content)) meta.content += ', viewport-fit=cover';
    if (!/interactive-widget/.test(meta.content)) meta.content += ', interactive-widget=resizes-content';
  }
  const iOS = /iP(hone|ad|od)/.test(navigator.userAgent) || (navigator.platform === 'MacIntel' && navigator.maxTouchPoints > 1);
  if (iOS && !/maximum-scale/.test(meta.content)) meta.content += ', maximum-scale=1';

  const readPref = () => { const v = store.get(PREF_KEY); return (v === 'mobile' || v === 'desktop') ? v : 'auto'; };
  let pref = readPref();
  const computeActive = () => pref === 'mobile' ? true : pref === 'desktop' ? false : mq.matches;

  const toggleBtn = mountTpl('mobile_toggle_btn');

  const syncToggleBtn = () => {
    const isMob = root.classList.contains('craft-mobile');
    const key = `${isMob ? 'm' : 'd'}:${pref}`;
    if (toggleBtn.dataset.state === key) return;
    const title = isMob
      ? `Switch to desktop view (current: ${pref === 'auto' ? 'auto mobile' : 'mobile'})`
      : `Switch to mobile view (current: ${pref === 'auto' ? 'auto desktop' : 'desktop'})`;
    patch(toggleBtn, {dataset: {state: key}, title, 'aria-label': title, 'aria-pressed': String(isMob)});
  };

  const mountToggleBtn = () => {
    if (!toggleBtn.isConnected && document.body) document.body.append(toggleBtn);
    syncToggleBtn();
  };

  const apply = () => {
    const on = computeActive();
    if (root.classList.contains('craft-mobile') !== on) {
      root.classList.toggle('craft-mobile', on);
      listeners.forEach(fn => { try { fn(on); } catch (err) { console.error('craftMobile listener', err); } });
    }
    syncToggleBtn();
  };

  const setMode = m => {
    pref = (m === 'mobile' || m === 'desktop') ? m : 'auto';
    store.set(PREF_KEY, pref === 'auto' ? null : pref);
    apply();
    return root.classList.contains('craft-mobile');
  };
  const toggle = () => setMode(root.classList.contains('craft-mobile') ? 'desktop' : 'mobile');
  on(toggleBtn, 'click', e => { e.preventDefault(); e.stopPropagation(); toggle(); }, sig);
  on(mq, 'change', apply, sig);
  apply();
  if (document.body) mountToggleBtn(); else on(document, 'DOMContentLoaded', mountToggleBtn, {once: true, signal: ac.signal});

  window.__hotkeyRegister?.({
    id: 'craft-toggle-mobile',
    label: 'Toggle mobile / desktop view',
    group: 'CRAFT · Layout',
    combo: '',
    handler: () => { toggle(); return true; },
  });

  window.craftMobile = {
    active: () => root.classList.contains('craft-mobile'),
    mode: () => pref,
    setMode,
    toggle,
    onChange: fn => { listeners.add(fn); return () => listeners.delete(fn); },
    reposition: () => {},
    disable() {
      ac.abort();
      toggleBtn.remove();
      root.classList.remove('craft-mobile');
      if (addedMeta) meta.remove(); else if (prevMeta != null) meta.content = prevMeta;
      listeners.clear();
      delete window.craftMobile;
    },
  };
})();

## Auto-Hiding Top Nav

Pure-CSS hover/focus reveal on `nav.monster-navbar` with a 700ms hide delay and dropdown/modal guards; the JS cell below only cleans up any hot-reloaded `window.craftAutoNav` listener from older builds.

In [ ]:
#| export_js CRAFT.js
#| eval: false
/* Auto-hiding top nav is now pure CSS in Dialog Chrome CSS below; clean up any hot-reloaded listener from older builds. */
window.craftAutoNav?.disable?.();
delete window.craftAutoNav;

## Compact Monaco Line Height

Aligns Solveit's native Monaco editors with `--craft-code` and `--craft-code-lh` so switching between inline CM6 and Monaco preserves visual density.

In [ ]:
#| export_js CRAFT.js
#| eval: false
/* Monaco is removed by craftNoMonaco below; clean up any hot-reloaded instance from older builds. */
window.craftCompactMonaco?.disable?.();

## Command & File Launcher (`Mod+Shift+P`)

Searchable palette combining registered `solveitHotkeys` actions with folders and dialogs already loaded by the file tree.

In [ ]:
#| export_js CRAFT.js
#| eval: false
/* ── Launcher (⌘⇧P) ──────────────────────────────────────────────────
   One fuzzy box over commands (every hotkey-registry action, custom and
   Solveit built-in) and dialogs/files (whatever the file tree has loaded).
   Enter runs or opens the selection; Esc or a click outside closes. */
(() => {
  window.craftLauncher?.disable?.();
  const {el, mountModal, qsa, on, notify, notifyErr, features} = window.__craftCMCore;
  const ac = new AbortController(), sig = {signal: ac.signal};
  let modal = null, results = [], sel = 0, pool = [];

  function score(q, text) {
    if (!q) return 1;
    const t = text.toLowerCase();
    let i = 0, s = 0, run = 0, last = -2;
    for (const ch of q) {
      const j = t.indexOf(ch, i);
      if (j < 0) return 0;
      run = j === last + 1 ? run + 1 : 0;
      s += 1 + run * 2 + (j === 0 || /[\s/_.-]/.test(t[j - 1]) ? 3 : 0);
      last = j; i = j + 1;
    }
    return s - t.length * 0.01;
  }

  const where = n => { try { const u = new URL(n.href, location.href); return u.searchParams.get('name') || u.searchParams.get('fp') || decodeURIComponent(u.pathname); } catch { return ''; } };

  function build() {
    modal = mountModal('launcher_root', {refs: true, signal: ac.signal});
    on(modal.input, {
      input: () => { sel = 0; render(); },
      keydown: e => {
        if (e.key === 'ArrowDown' || e.key === 'ArrowUp') { e.preventDefault(); e.stopPropagation(); sel = (sel + (e.key === 'ArrowDown' ? 1 : -1) + results.length) % Math.max(results.length, 1); paint(); }
        else if (e.key === 'Enter') { e.preventDefault(); e.stopPropagation(); run(results[sel]); }
      },
    }, sig);
    on(modal.list, 'click', e => { const li = e.target.closest('li[data-i]'); if (li) run(results[+li.dataset.i]); }, sig);
  }

  function render() {
    const q = modal.input.value.trim().toLowerCase();
    results = pool.map(it => ({it, s: score(q, it.label + ' ' + it.detail)})).filter(r => r.s > 0).sort((a, b) => b.s - a.s).slice(0, 60).map(r => r.it);
    sel = Math.min(sel, Math.max(results.length - 1, 0));
    modal.list.replaceChildren(...(results.length ? results.map((it, i) =>
      el('li', {cls: 'kind-' + it.kind, role: 'option', dataset: {i}},
        el('span', {cls: 'label'}, it.label), el('span', {cls: 'detail'}, it.detail), el('span', {cls: 'hint'}, it.hint))
    ) : [el('li', {cls: 'empty'}, 'No matches')]));
    paint();
  }

  function paint() {
    qsa('li[data-i]', modal.list).forEach(li => { const on = +li.dataset.i === sel; li.setAttribute('aria-selected', String(on)); if (on) li.scrollIntoView({block: 'nearest'}); });
  }

  async function open() {
    if (!modal) build();
    const hk = window.solveitHotkeys;
    const cmds = (hk?.list?.() || []).filter(a => a.id !== 'launcher').map(a => ({kind: 'command', label: a.label, detail: a.group || '', hint: a.combo ? hk.formatCombo(a.combo) : '', go: () => hk.run(a.id)}));
    const toggles = (features?.list?.() || []).map(f => ({
      kind: 'feature',
      label: `Feature: ${f.label}`,
      detail: f.desc || f.id,
      hint: f.enabled ? 'On' : 'Off',
      go: () => {
        const next = features.toggle(f.id);
        notify(`${f.label}: ${next ? 'enabled' : 'disabled'} — reload page to apply.`, {status: next ? 'success' : 'warning'});
      },
    }));
    const base = cmds.concat(toggles);
    pool = base; modal.input.value = ''; sel = 0; render(); modal.open(); modal.input.focus();
    try {
      const entries = await window.solveitFileTree?.entries?.() || [];
      pool = base.concat(entries.map(n => ({kind: n.kind, label: n.name, detail: where(n), hint: n.kind, go: () => window.solveitFileTree.open(n)})));
      if (modal.isOpen()) render();
    } catch {}
  }

  const close = () => modal?.close();
  function run(item) { if (!item) return; close(); setTimeout(() => Promise.resolve().then(item.go).catch(err => { console.error('Launcher action failed', err); notifyErr('Action failed: ' + (err.message || err), 'craftLauncher'); }), 0); }

  window.__hotkeyRegister?.({id: 'launcher', label: 'Open launcher', group: 'Launcher', combo: 'Mod+Shift+P', handler: () => { if (modal?.isOpen()) close(); else open(); return true; }});
  window.craftLauncher = { open, close, disable() { ac.abort(); modal?.destroy(); modal = null; } };
})();

## Disable Monaco & Remove Bottom Composer (`window.craftNoMonaco`)

Unconditionally hides and removes `#full_editor`, disposes existing Monaco editors and models, installs no-op `window.monaco` and `window.editor` stubs, and blocks HTMX swaps into `#full_editor`.

In [ ]:
#| export_js CRAFT.js
#| eval: false
/* ── Remove Monaco & #full_editor unconditionally ─────────────────── */
(() => {
  window.craftIdleComposer?.disable?.();
  window.craftNoMonaco?.disable?.();
  const {on} = window.__craftCMCore;
  const ac = new AbortController(), sig = {signal: ac.signal};
  let retryTimer = null, tries = 0, disposedReal = false;

  const MONACO_STUB = Object.freeze({
    __craftStub: true,
    editor: Object.freeze({
      getEditors: () => [],
      getModels: () => [],
      getDiffEditors: () => [],
      setTheme() {},
      remeasureFonts() {},
      onDidCreateEditor: () => ({dispose() {}}),
    }),
  });

  const stubContainer = document.createElement('div');
  const EDITOR_STUB = Object.freeze({
    __craftStub: true,
    getContainerDomNode: () => stubContainer,
    getDomNode: () => stubContainer,
    layout() {},
    focus() {},
    hasTextFocus: () => false,
    hasWidgetFocus: () => false,
    getModel: () => null,
    setModel() {},
    getValue: () => '',
    setValue() {},
    dispose() {},
    updateOptions() {},
    getRawOptions: () => ({}),
    getOption: () => undefined,
    getPosition: () => null,
    getSelection: () => null,
    onDidChangeModel: () => ({dispose() {}}),
    onDidChangeModelContent: () => ({dispose() {}}),
    trigger() {},
  });

  const wrapFollowRename = () => {
    const fn = window.followRename;
    if (typeof fn !== 'function' || fn.__craftSafe) return;
    const wrapped = function(...args) {
      if (!document.getElementById('id_')) return;
      return fn.apply(this, args);
    };
    wrapped.__craftSafe = true;
    window.followRename = wrapped;
  };

  const disposeMonaco = (forceRemove = false) => {
    const hasRealMonaco = Boolean(window.monaco && !window.monaco.__craftStub);
    const hasRealEditor = Boolean(window.editor && !window.editor.__craftStub);
    const hasEditors = Boolean(hasRealEditor || window.diffEditor || (hasRealMonaco && window.monaco.editor?.getEditors?.().length));
    document.documentElement.classList.remove('craft-show-monaco');
    if (!hasEditors && !forceRemove) return false;
    if (hasRealMonaco || hasEditors) {
      try { window.cleanupEd?.(); } catch {}
      try { if (hasRealEditor) window.editor?.dispose?.(); } catch {}
      try { window.diffEditor?.dispose?.(); } catch {}
      try { window.monaco?.editor?.getEditors?.().forEach(ed => { try { ed.dispose(); } catch {} }); } catch {}
      try { window.monaco?.editor?.getDiffEditors?.().forEach(ed => { try { ed.dispose(); } catch {} }); } catch {}
      try { window.monaco?.editor?.getModels?.().forEach(m => { try { m.dispose(); } catch {} }); } catch {}
      window.diffEditor = null;
    }
    window.editor = EDITOR_STUB;
    window.monaco = MONACO_STUB;
    window.createMonacoEditor = () => null;
    window.msgToModel = () => null;
    window.pushModelEdits = () => {};
    window.edValue = () => '';
    wrapFollowRename();
    document.getElementById('full_editor')?.remove();
    disposedReal = true;
    return true;
  };

  const blockFullEditorSwap = e => {
    const tgt = e.detail?.target || e.target;
    if (tgt?.id === 'full_editor' || tgt?.closest?.('#full_editor')) {
      if (e.detail) e.detail.shouldSwap = false;
      e.preventDefault();
      if (disposedReal) document.getElementById('full_editor')?.remove();
    }
  };

  const sep = window.htmx?.config?.metaCharacter || ':';
  on(document, ['htmx', 'oobBeforeSwap'].join(sep), blockFullEditorSwap, sig);
  on(document, ['htmx', 'beforeSwap'].join(sep), blockFullEditorSwap, sig);
  on(window, 'load', () => disposeMonaco(false), {once: true, signal: ac.signal});

  // Wait for in-flight initMonaco()/_cme() to create window.editor before disposing & removing #full_editor,
  // with a bounded fallback so non-Monaco pages still clean up without leaving a permanent timer.
  const ensureDisposed = () => {
    if (ac.signal.aborted) return;
    if (disposeMonaco(false)) return;
    if (++tries < 40) {
      retryTimer = setTimeout(ensureDisposed, 100);
    } else {
      disposeMonaco(true);
    }
  };
  ensureDisposed();

  const teardown = () => disposeMonaco(true);
  window.craftNoMonaco = {
    teardown,
    disable() {
      ac.abort();
      clearTimeout(retryTimer);
      retryTimer = null;
      if (window.craftNoMonaco?.teardown === teardown) delete window.craftNoMonaco;
    },
  };
})();

## Accessibility Pass (`window.craftA11y`)

Mirrors Solveit `uk-tooltip` attributes into `aria-label`s on icon-only buttons, `#find-box` controls, and navbar controls so screen readers and keyboard users get meaningful labels.

In [ ]:
#| export_js CRAFT.js
#| eval: false
/* ── Accessibility: mirror uk-tooltip & fallback aria-labels on icon controls ── */
(() => {
  window.craftA11y?.disable?.();
  if (!window.__craftCMCore.features.init('a11y', 'Accessibility labels', 'Mirror uk-tooltip into aria-label on icon controls')) return;
  const {SOLVEIT, onDomChange} = window.__craftCMCore;
  const extractTip = raw => {
    if (!raw) return '';
    const s = String(raw).trim();
    const m = s.match(/(?:^|;)\s*title\s*:\s*([^;]+)/i);
    return (m ? m[1] : s).replace(/<[^>]*>/g, '').trim();
  };
  const labelControls = (root = document) => {
    // 1. Any interactive element carrying uk-tooltip / data-uk-tooltip without an accessible name
    for (const el of root.querySelectorAll('[uk-tooltip], [data-uk-tooltip]')) {
      if (!el.matches('button, a, input, select, textarea')) continue;
      if (el.hasAttribute('aria-label') || el.hasAttribute('aria-labelledby')) continue;
      const tip = extractTip(el.getAttribute('uk-tooltip') || el.getAttribute('data-uk-tooltip'));
      if (tip) el.setAttribute('aria-label', tip);
    }
    // 2. #find-box controls (checkboxes wrapped in <label uk-tooltip>, search/replace inputs)
    const fb = root.querySelector?.(SOLVEIT.sel.findBox) || document.querySelector(SOLVEIT.sel.findBox);
    if (fb) {
      for (const el of fb.querySelectorAll('input, button, select')) {
        if (el.hasAttribute('aria-label') || el.hasAttribute('aria-labelledby')) continue;
        const lbl = el.closest('label') || (el.id && fb.querySelector(`label[for="${CSS.escape(el.id)}"]`));
        const tip = extractTip(
          el.getAttribute('uk-tooltip') || el.getAttribute('data-uk-tooltip') ||
          lbl?.getAttribute('uk-tooltip') || lbl?.getAttribute('data-uk-tooltip') ||
          el.getAttribute('title') || lbl?.getAttribute('title')
        );
        const fallback = tip || (el.getAttribute('placeholder') || '').replace(/\.+$/, '').trim();
        if (fallback) el.setAttribute('aria-label', fallback);
      }
    }
    // 3. Navbar <select name="mode"> and Dialogs link
    const nav = root.querySelector?.('.uk-container > nav.monster-navbar') || document.querySelector('.uk-container > nav.monster-navbar');
    if (nav) {
      const modeSel = nav.querySelector('select[name="mode"]');
      if (modeSel && !modeSel.hasAttribute('aria-label')) modeSel.setAttribute('aria-label', 'AI mode');
      const dlgsLink = nav.querySelector('a[href^="/?at="]');
      if (dlgsLink && !dlgsLink.hasAttribute('aria-label') && !(dlgsLink.textContent || '').trim()) {
        dlgsLink.setAttribute('aria-label', 'Dialogs');
      }
      // 4. Any remaining icon-only navbar buttons/links without accessible name
      for (const btn of nav.querySelectorAll('button, a')) {
        if (btn.hasAttribute('aria-label') || btn.hasAttribute('aria-labelledby')) continue;
        if ((btn.textContent || '').trim()) continue;
        const tip = extractTip(
          btn.getAttribute('title') || btn.getAttribute('uk-tooltip') || btn.getAttribute('data-uk-tooltip')
        );
        const iconRef = btn.querySelector('use')?.getAttribute('href')?.replace(/^#lc-/, '') ||
                        btn.getAttribute('uk-icon') || btn.querySelector('[uk-icon]')?.getAttribute('uk-icon') || '';
        const hxPost = (btn.getAttribute('hx-post') || btn.getAttribute('hx-get') || '').replace(/^\/|_+$/g, '').replace(/_/g, ' ');
        const fallback = tip || (iconRef ? iconRef.replace(/[-_]+/g, ' ').replace(/^\w/, c => c.toUpperCase()) : '') || hxPost;
        if (fallback) btn.setAttribute('aria-label', fallback);
      }
    }
  };
  let offDom = null;
  const start = () => {
    labelControls(document);
    offDom = onDomChange(() => labelControls(document), {delay: 120});
  };
  if (document.readyState === 'loading') document.addEventListener('DOMContentLoaded', start);
  else start();
  window.craftA11y = { refresh: () => labelControls(document), disable() { offDom?.(); offDom = null; document.removeEventListener('DOMContentLoaded', start); } };
})();

## Dialog Chrome CSS

Styles the auto-hiding navbar, top-right button reservation, flush bottom composer, `#solveit-launcher`, and `#craft-mobile-toggle`.

In [ ]:
#| export_css CRAFT.css
#| eval: false
/* Reserve top-right space inside desktop top nav so #craft-mobile-toggle never overlaps navbar buttons. */
html:not(.craft-mobile) .uk-container > nav.monster-navbar {
  padding-right: 44px !important;
  box-sizing: border-box;
}
html:not(.craft-mobile) .uk-container > nav.monster-navbar > div.ml-auto {
  display: flex !important;
  flex-wrap: nowrap !important;
  align-items: center;
  gap: 8px;
  height: 100%;
}

/* ── 7. Top nav: pure-CSS auto-hiding overlay, floating controls ────── */
.uk-container > nav.monster-navbar {
  position: absolute !important;         /* out of flow: scroller keeps the 44px */
  top: 0; left: 0; right: 0;
  z-index: 40;
  height: 31px !important;               /* 0.7 x original */
  min-height: 0 !important;
  flex-wrap: nowrap !important;
  padding-block: 0 !important;
  background: transparent !important;
  border: 0 !important;
  opacity: 0;
  pointer-events: none;
  transition: opacity 0.18s ease 0.7s;   /* 700ms grace period before fading out */
}
html:not(.craft-mobile) .uk-container > nav.monster-navbar { pointer-events: auto; }
html:not(.craft-mobile) .uk-container > nav.monster-navbar:is(:hover, :focus-within, .nav-shown),
html:not(.craft-mobile) body:has(.uk-drop.uk-open, .uk-dropdown.uk-open) .uk-container > nav.monster-navbar,
.uk-container > nav.monster-navbar:is(.nav-shown, :focus-within) {
  opacity: 1;
  pointer-events: auto;
  transition-delay: 0s;
}
.uk-container > nav.monster-navbar > * { pointer-events: none; }
html:not(.craft-mobile) .uk-container > nav.monster-navbar:is(:hover, :focus-within, .nav-shown) > *,
html:not(.craft-mobile) body:has(.uk-drop.uk-open, .uk-dropdown.uk-open) .uk-container > nav.monster-navbar > *,
.uk-container > nav.monster-navbar:is(.nav-shown, :focus-within) > * { pointer-events: auto; }
body:has(dialog.craft-overlay[open], .uk-modal.uk-open) .uk-container > nav.monster-navbar {
  opacity: 0 !important;
  pointer-events: none !important;
}

/* Bottom Monaco composer: always hidden before craftNoMonaco removes it from the DOM */
#full_editor { display: none !important; }

/* Desktop mobile-view toggle positioning and icon swap (pure CSS) */
html:not(.craft-mobile) #craft-mobile-toggle { left: auto; right: 10px; top: 6px; }
html.craft-mobile #craft-mobile-toggle .craft-icon-phone,
html:not(.craft-mobile) #craft-mobile-toggle .craft-icon-desk { display: none !important; }
.uk-container > nav.monster-navbar h2 { font-size: 1.05rem !important; line-height: 1.2 !important; }
.uk-container > nav.monster-navbar span.rounded-full.size-4,
.uk-container > nav.monster-navbar a:has(> h1),
.uk-container > nav.monster-navbar h2#dlg-name { display: none !important; }

/* Strip action wrapper, chips (UIkit paints them as outlines), and <select> frame */
.uk-container > nav.monster-navbar :is(> div.ml-auto, .uk-btn, .uk-btn-group, .uk-btn-group > *, select, input:checked + label.uk-btn, .uk-btn.uk-active) {
  background: transparent !important;
  border: 0 !important;
  border-radius: 0 !important;
  box-shadow: none !important;
  outline: 0 !important;
  padding: 0 !important;
}
.uk-container > nav.monster-navbar :is(.uk-btn, select) { color: var(--craft-fg) !important; }
.uk-container > nav.monster-navbar select { padding-inline: 2px !important; }
.uk-container > nav.monster-navbar .uk-btn:hover { color: var(--craft-fg-strong) !important; }
.uk-container > nav.monster-navbar :is(label, button):has(input:checked) {
  color: var(--craft-accent) !important;
  filter: drop-shadow(0 0 4px #7aa2f7cc) drop-shadow(0 0 10px #7aa2f766);
}
.uk-container > nav.monster-navbar .uk-btn.uk-active { color: var(--craft-accent) !important; }
.uk-container > nav.monster-navbar select option { background: var(--craft-recessed) !important; color: var(--craft-fg) !important; }
.uk-container > nav.monster-navbar :is(svg, select, h2) { filter: drop-shadow(0 1px 2px var(--craft-recessed)) drop-shadow(0 0 4px var(--craft-recessed)); }

/* ── Mobile top nav: hidden until Menu, then a solid wrapping bar housing the view toggle ── */
html.craft-mobile .uk-container > nav.monster-navbar {
  position: fixed !important; top: 0; left: 0; right: 0; z-index: 960;
  height: auto !important; min-height: 48px !important;
  display: flex !important; flex-wrap: wrap !important; align-items: center; gap: 6px 14px;
  padding: calc(6px + env(safe-area-inset-top, 0px)) 54px 6px 12px !important;
  background: var(--craft-recessed) !important; border-bottom: 1px solid #ffffff14 !important;
  box-shadow: 0 8px 24px #0009;
  opacity: 0; pointer-events: none; transform: translateY(-100%);
  transition: opacity .16s ease, transform .16s ease;
}
html.craft-mobile.craft-nav-open .uk-container > nav.monster-navbar,
html.craft-mobile .uk-container > nav.monster-navbar:focus-within { opacity: 1; pointer-events: auto; transform: none; }
html.craft-mobile.craft-nav-open .uk-container > nav.monster-navbar > * { pointer-events: auto; }
html.craft-mobile .uk-container > nav.monster-navbar > div.min-w-0,
html.craft-mobile .uk-container > nav.monster-navbar > button { display: none !important; }
html.craft-mobile .uk-container > nav.monster-navbar > div.ml-auto { display: flex !important; flex-wrap: wrap; align-items: center; gap: 6px 12px; width: 100%; margin: 0 !important; padding: 0 !important; }
html.craft-mobile .uk-container > nav.monster-navbar > div.ml-auto > div { flex: 1 1 100%; min-width: 0; justify-content: flex-start !important; gap: 6px 8px !important; }
html.craft-mobile .uk-container > nav.monster-navbar .uk-btn-group { flex-wrap: wrap !important; max-width: 100%; }
html.craft-mobile .uk-container > nav.monster-navbar :is(.uk-btn, label, button, select) { min-height: 36px; min-width: 36px; font-size: 14px !important; }
html.craft-mobile .uk-container > nav.monster-navbar svg,
html.craft-mobile .uk-container > nav.monster-navbar select,
html.craft-mobile .uk-container > nav.monster-navbar h2 { filter: none; }

/* ── Launcher (⌘⇧P) ─────────────────────────────────────────────────── */
#solveit-launcher .solveit-launcher-card { width: min(640px, 92vw); background: var(--craft-bg); border-radius: 8px; overflow: hidden; }
#solveit-launcher input { box-sizing: border-box; width: 100%; padding: 10px 14px; border: 0; border-bottom: 1px solid #ffffff14; outline: 0; background: transparent; color: var(--craft-fg); font-size: 14px; }
#solveit-launcher ul { max-height: 50vh; overflow-y: auto; margin: 0; padding: 4px; list-style: none; }
#solveit-launcher li { display: flex; align-items: baseline; gap: 10px; padding: 5px 10px; border-radius: 4px; color: var(--craft-fg); font-size: 12.5px; cursor: pointer; }
#solveit-launcher li[aria-selected="true"] { background: var(--craft-hover); }
#solveit-launcher li .label { flex: 0 1 auto; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
#solveit-launcher li .detail { flex: 1 1 0%; min-width: 0; color: var(--craft-muted-soft); white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
#solveit-launcher li .hint { flex: 0 0 auto; color: var(--craft-muted); font: 11px monospace; }
#solveit-launcher li.kind-dialog .hint { color: var(--craft-purple); }
#solveit-launcher li.empty { color: var(--craft-muted-soft); cursor: default; }

/* ── Shared floating top-right nav icon buttons (#craft-mobile-toggle & #craft-insert-preview) ── */
.craft-nav-icon-btn {
  position: fixed; z-index: 955; width: 28px; height: 26px; display: grid; place-items: center;
  border: 0; border-radius: 4px; background: transparent; color: #9b9cab; opacity: .4; padding: 0;
  cursor: pointer; transition: opacity .15s ease, color .15s ease;
}
.craft-nav-icon-btn:is(:hover, :focus-visible),
body:has(nav.monster-navbar:is(:hover, :focus-within, .nav-shown), .uk-drop.uk-open, .uk-dropdown.uk-open) .craft-nav-icon-btn { opacity: 1; color: #e0e0e8; }
.craft-nav-icon-btn:focus-visible { outline: 1px solid var(--craft-accent); outline-offset: 2px; }

/* Move the desktop/mobile toggle into the revealed top nav instead of floating over cells. */
html.craft-mobile #craft-mobile-toggle {
  top: calc(8px + env(safe-area-inset-top, 0px));
  left: auto; right: 10px; z-index: 970;
  width: 36px; height: 36px; border-radius: 8px;
  background: #242638; border: 1px solid #ffffff28;
  color: var(--craft-fg);
  opacity: 0; pointer-events: none; transform: translateY(-100%);
  transition: opacity .16s ease, transform .16s ease;
}
html.craft-mobile.craft-nav-open #craft-mobile-toggle {
  opacity: 1; pointer-events: auto; transform: none;
}
html.craft-mobile #solveit-launcher { padding-top: calc(10px + env(safe-area-inset-top, 0px)); }
html.craft-mobile #solveit-launcher .solveit-launcher-card { width: calc(100vw - 16px); }
html.craft-mobile #solveit-launcher input { font-size: 16px; padding: 14px; }
html.craft-mobile #solveit-launcher ul { max-height: 60dvh; }
html.craft-mobile #solveit-launcher li { padding: 11px 12px; font-size: 14px; }
html.craft-mobile #solveit-launcher li .hint { display: none; }

Verify `05_dialog_chrome.ipynb` exports its templates, JS modules, and CSS cleanly.

In [ ]:
from craft_dev.auth import REPO_ROOT
from craft_dev.build import extract_nb_assets

assets = extract_nb_assets(REPO_ROOT/'nbs/craft/05_dialog_chrome.ipynb')
js = '\n'.join(code for _, code in assets['CRAFT.js'])
for sym in ('window.craftMobile', 'window.craftNoMonaco', 'window.craftA11y', 'solveit-launcher'):
    assert sym in js, f"Missing {sym}"
assert '#solveit-launcher' in '\n'.join(code for _, code in assets['CRAFT.css'])

## Live `fastcdp` Verification

In [ ]:
#| live
from fastcdp import Rungs
from craft_dev.cdp import open_solveit

async with open_solveit() as (page, _, page_errors):
    rungs = Rungs(page)

    async with rungs('command launcher open, filter, close'):
        await page.press('P', mod=True, shift=True)
        await page.wait_for_selector('#solveit-launcher[open] input', timeout=3)
        await page.fill_text('#solveit-launcher[open] input', 'line')
        await page.wait_for("document.querySelectorAll('#solveit-launcher[open] ul[role=\"listbox\"] li').length >= 1", timeout=3)
        await page.press('Escape')
        await page.wait_for_selector('#solveit-launcher[open]', present=False, timeout=3)

    async with rungs('desktop/mobile mode toggle'):
        assert await page.eval("window.craftMobile.active() === false")
        assert await page.sel_exists('#craft-mobile-toggle')
        await page.eval("window.craftMobile.toggle()")
        assert await page.eval("window.craftMobile.active() === true")
        await page.eval("window.craftMobile.setMode('auto')")
        assert await page.eval("window.craftMobile.active() === false")

    assert not page_errors, f"Uncaught page errors: {page_errors}"